## Load the CSV

In [1]:
import numpy as np
import pandas as pd

In [2]:
df = pd.read_csv('data/Walmart.csv')
df.head()

,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment
0,1,05-02-2010,1643690.90,0,42.31,2.572,211.096358,8.106
1,1,12-02-2010,1641957.44,1,38.51,2.548,211.242170,8.106
2,1,19-02-2010,1611968.17,0,39.93,2.514,211.289143,8.106
3,1,26-02-2010,1409727.59,0,46.63,2.561,211.319643,8.106
4,1,05-03-2010,1554806.68,0,46.50,2.625,211.350143,8.106


In [3]:
df.shape

(6435, 8)

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6435 entries, 0 to 6434
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Store         6435 non-null   int64  
 1   Date          6435 non-null   object 
 2   Weekly_Sales  6435 non-null   float64
 3   Holiday_Flag  6435 non-null   int64  
 4   Temperature   6435 non-null   float64
 5   Fuel_Price    6435 non-null   float64
 6   CPI           6435 non-null   float64
 7   Unemployment  6435 non-null   float64
dtypes: float64(5), int64(2), object(1)
memory usage: 402.3+ KB


In [5]:
df.columns

Index(['Store', 'Date', 'Weekly_Sales', 'Holiday_Flag', 'Temperature',
       'Fuel_Price', 'CPI', 'Unemployment'],
      dtype='object')

## Load the document

In [6]:
from langchain_community.document_loaders.csv_loader import CSVLoader
loader = CSVLoader(file_path='data/Walmart.csv')
data = loader.load()

/var/folders/wt/f5tjfmdj3p550q42d1n61syw0000gn/T/ipykernel_88738/2019483659.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders.csv_loader import CSVLoader


In [7]:
print("Type of loaded data:", type(data))

print("Number of datapoints:", len(data))

print("Type of each datapoints:", type(data[0]))

Type of loaded data: <class 'list'>
Number of datapoints: 6435
Type of each datapoints: <class 'langchain_core.documents.base.Document'>


In [8]:
data[:5]

[Document(metadata={'source': 'data/Walmart.csv', 'row': 0}, page_content='Store: 1\nDate: 05-02-2010\nWeekly_Sales: 1643690.9\nHoliday_Flag: 0\nTemperature: 42.31\nFuel_Price: 2.572\nCPI: 211.0963582\nUnemployment: 8.106'),
 Document(metadata={'source': 'data/Walmart.csv', 'row': 1}, page_content='Store: 1\nDate: 12-02-2010\nWeekly_Sales: 1641957.44\nHoliday_Flag: 1\nTemperature: 38.51\nFuel_Price: 2.548\nCPI: 211.2421698\nUnemployment: 8.106'),
 Document(metadata={'source': 'data/Walmart.csv', 'row': 2}, page_content='Store: 1\nDate: 19-02-2010\nWeekly_Sales: 1611968.17\nHoliday_Flag: 0\nTemperature: 39.93\nFuel_Price: 2.514\nCPI: 211.2891429\nUnemployment: 8.106'),
 Document(metadata={'source': 'data/Walmart.csv', 'row': 3}, page_content='Store: 1\nDate: 26-02-2010\nWeekly_Sales: 1409727.59\nHoliday_Flag: 0\nTemperature: 46.63\nFuel_Price: 2.561\nCPI: 211.3196429\nUnemployment: 8.106'),
 Document(metadata={'source': 'data/Walmart.csv', 'row': 4}, page_content='Store: 1\nDate: 05-03-

In [9]:
print(data[0].page_content)

Store: 1
Date: 05-02-2010
Weekly_Sales: 1643690.9
Holiday_Flag: 0
Temperature: 42.31
Fuel_Price: 2.572
CPI: 211.0963582
Unemployment: 8.106


Chunking not needed for this data because the data is small and it would make it worse. Instead we can improve metadata

## Improving Metadata and page content

In [10]:
from langchain_core.documents import Document

new_data = []

for doc in data:

    lines = doc.page_content.split("\n")

    store = lines[0].split(": ")[1]
    date = lines[1].split(": ")[1]
    sales = lines[2].split(": ")[1]
    holiday = lines[3].split(": ")[1]
    temp = lines[4].split(": ")[1]
    fuel = lines[5].split(": ")[1]
    cpi = lines[6].split(": ")[1]
    unemployment = lines[7].split(": ")[1]


    # new cleaner page content
    new_content = f"""
Store {store} recorded weekly sales of {sales}.
The date was {date}.
This was a holiday week: {holiday}.
The temperature was {temp}.
Fuel price was {fuel}.
CPI was {cpi}.
Unemployment rate was {unemployment}.
""".strip()


    # metadata
    metadata = {
        "store": store,
        "date": date,
        "holiday": holiday
    }


    new_doc = Document(
        page_content=new_content,
        metadata=metadata
    )

    new_data.append(new_doc)

In [11]:
new_data[0]

Document(metadata={'store': '1', 'date': '05-02-2010', 'holiday': '0'}, page_content='Store 1 recorded weekly sales of 1643690.9.\nThe date was 05-02-2010.\nThis was a holiday week: 0.\nThe temperature was 42.31.\nFuel price was 2.572.\nCPI was 211.0963582.\nUnemployment rate was 8.106.')

In [12]:
print(new_data[0].page_content)

Store 1 recorded weekly sales of 1643690.9.
The date was 05-02-2010.
This was a holiday week: 0.
The temperature was 42.31.
Fuel price was 2.572.
CPI was 211.0963582.
Unemployment rate was 8.106.


## Embedding

In [13]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

In [14]:
new_data[:7]

[Document(metadata={'store': '1', 'date': '05-02-2010', 'holiday': '0'}, page_content='Store 1 recorded weekly sales of 1643690.9.\nThe date was 05-02-2010.\nThis was a holiday week: 0.\nThe temperature was 42.31.\nFuel price was 2.572.\nCPI was 211.0963582.\nUnemployment rate was 8.106.'),
 Document(metadata={'store': '1', 'date': '12-02-2010', 'holiday': '1'}, page_content='Store 1 recorded weekly sales of 1641957.44.\nThe date was 12-02-2010.\nThis was a holiday week: 1.\nThe temperature was 38.51.\nFuel price was 2.548.\nCPI was 211.2421698.\nUnemployment rate was 8.106.'),
 Document(metadata={'store': '1', 'date': '19-02-2010', 'holiday': '0'}, page_content='Store 1 recorded weekly sales of 1611968.17.\nThe date was 19-02-2010.\nThis was a holiday week: 0.\nThe temperature was 39.93.\nFuel price was 2.514.\nCPI was 211.2891429.\nUnemployment rate was 8.106.'),
 Document(metadata={'store': '1', 'date': '26-02-2010', 'holiday': '0'}, page_content='Store 1 recorded weekly sales of 14

In [15]:
vectors = embedding_model.embed_documents(
    [doc.page_content for doc in new_data]
)

In [16]:
print(len(vectors))
print(vectors[0][:5])

6435
[-0.04333607852458954, 0.07548099756240845, -0.009466942399740219, 0.0809648334980011, -0.05234026536345482]


## Create vector store

In [17]:
from langchain_community.vectorstores import FAISS

In [18]:
vectorstore = FAISS.from_documents(
    documents=new_data,
    embedding=embedding_model
)

In [19]:
query = "Which store had high sales during holidays?"

results = vectorstore.similarity_search(query, k=3)

In [20]:
for r in results:
    print(r.page_content)
    print(r.metadata)
    print("-----")

Store 10 recorded weekly sales of 1933469.15.
The date was 11-03-2011.
This was a holiday week: 0.
The temperature was 64.22.
Fuel price was 3.63.
CPI was 128.3995.
Unemployment rate was 8.744.
{'store': '10', 'date': '11-03-2011', 'holiday': '0'}
-----
Store 10 recorded weekly sales of 1917397.63.
The date was 12-08-2011.
This was a holiday week: 0.
The temperature was 85.61.
Fuel price was 3.794.
CPI was 129.2015806.
Unemployment rate was 8.257.
{'store': '10', 'date': '12-08-2011', 'holiday': '0'}
-----
Store 27 recorded weekly sales of 1688955.49.
The date was 29-10-2010.
This was a holiday week: 0.
The temperature was 61.02.
Fuel price was 3.055.
CPI was 136.7375484.
Unemployment rate was 8.021.
{'store': '27', 'date': '29-10-2010', 'holiday': '0'}
-----


## Hybrid Retriever

In [21]:
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

# FAISS retriever
faiss_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# BM25 retriever
bm25_retriever = BM25Retriever.from_documents(new_data)
bm25_retriever.k = 3

# Hybrid retriever
hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, faiss_retriever],
    weights=[0.4, 0.6]
)

In [22]:
query = "Store 5 sales during holiday in 2012"

results = hybrid_retriever.invoke(query)

for i, r in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(r.page_content)


--- Result 1 ---
Store 5 recorded weekly sales of 353652.23.
The date was 04-11-2011.
This was a holiday week: 0.
The temperature was 56.71.
Fuel price was 3.332.
CPI was 218.408408.
Unemployment rate was 6.3.

--- Result 2 ---
Store 5 recorded weekly sales of 291454.52.
The date was 13-01-2012.
This was a holiday week: 0.
The temperature was 48.86.
Fuel price was 3.261.
CPI was 220.4760185.
Unemployment rate was 5.943.

--- Result 3 ---
Store 5 recorded weekly sales of 325345.41.
The date was 12-10-2012.
This was a holiday week: 0.
The temperature was 66.24.
Fuel price was 3.601.
CPI was 223.974787.
Unemployment rate was 5.422.

--- Result 4 ---
Store 5 recorded weekly sales of 321299.99.
The date was 23-03-2012.
This was a holiday week: 0.
The temperature was 64.44.
Fuel price was 3.787.
CPI was 221.8735063.
Unemployment rate was 5.943.

--- Result 5 ---
Store 5 recorded weekly sales of 292539.73.
The date was 22-07-2011.
This was a holiday week: 0.
The temperature was 86.97.
Fuel p

## Chat Prompt template

In [23]:
from langchain_core.prompts import ChatPromptTemplate

PROMPT_TEMPLATE = """
You are a Walmart business analyst.

Answer the question based only on the following context:

{context}

Question:
{question}

Provide a clear and concise answer.
Do not include any information not present in the context.
"""

prompt_template = ChatPromptTemplate.from_template(PROMPT_TEMPLATE)

## Initialize the Generator (Chat Model)

In [27]:
f = open('keys/openai_api_key.txt')
OPENAI_API_KEY = f.read()

In [28]:
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(
    api_key=OPENAI_API_KEY,
    model="gpt-4o-mini",
    temperature=0
)

## Initialize Output Parser

In [29]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

## Define a RAG chain

In [30]:
from langchain_core.runnables import RunnablePassthrough

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [32]:
rag_chain = {"context": hybrid_retriever | format_docs, "question": RunnablePassthrough()} | prompt_template | chat_model | parser

## Invoke the chain

In [36]:
query = "What factors affect weekly sales in Walmart stores?"

response = rag_chain.invoke(query)
print(response)

The factors that affect weekly sales in Walmart stores, based on the provided context, include:

1. **Temperature**: Variations in temperature can influence customer shopping behavior.
2. **Fuel Price**: Changes in fuel prices may impact consumer spending and travel to the store.
3. **Consumer Price Index (CPI)**: The CPI reflects inflation and purchasing power, which can affect sales.
4. **Unemployment Rate**: Higher unemployment may lead to reduced consumer spending, impacting sales.
5. **Holiday Weeks**: Sales may be influenced by whether the week is a holiday week or not. 

All these factors can contribute to fluctuations in weekly sales figures.


## Testing

In [37]:
ground_truth_data = [
    {
        "query": "Do sales increase during holiday weeks?",
        "expected": "Sales are higher during holiday weeks"
    },
    {
        "query": "What is the effect of unemployment on sales?",
        "expected": "Higher unemployment reduces sales"
    },
    {
        "query": "How does fuel price affect sales?",
        "expected": "Fuel price changes impact sales slightly"
    },
    {
        "query": "Which factor strongly influences weekly sales?",
        "expected": "Holiday flag and store type influence sales the most"
    }
]

In [42]:
# Run evaluation

results = []

for item in ground_truth_data:
    query = item["query"]
    expected = item["expected"]

    try:
        predicted = rag_chain.invoke(query)
    except Exception as e:
        predicted = str(e)

    results.append({
        "query": query,
        "expected": expected,
        "predicted": predicted
    })

In [43]:
# Convert to dataframe 

import pandas as pd

results_df = pd.DataFrame(results)
results_df

,query,expected,predicted
0,Do sales increase during holiday weeks?,Sales are higher during holiday weeks,"Yes, sales increase during holiday weeks, as e..."
1,What is the effect of unemployment on sales?,Higher unemployment reduces sales,"Based on the provided data, there is a general..."
2,How does fuel price affect sales?,Fuel price changes impact sales slightly,"Based on the provided data, there is no clear ..."
3,Which factor strongly influences weekly sales?,Holiday flag and store type influence sales th...,The context does not provide a definitive answ...


In [44]:
# Semantic scoring

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [45]:
# Loading embedding model

model = SentenceTransformer("all-MiniLM-L6-v2")

In [46]:
# Similarity search

def similarity(a, b):
    embeddings = model.encode([a, b])
    return cosine_similarity([embeddings[0]], [embeddings[1]])[0][0]

In [47]:
# Apply scoring

results_df["score"] = results_df.apply(
    lambda row: similarity(row["expected"], row["predicted"]),
    axis=1
)

In [48]:
# Final evaluation summary

print("Average RAG Score:", results_df["score"].mean())
print("Min Score:", results_df["score"].min())
print("Max Score:", results_df["score"].max())

Average RAG Score: 0.72490215
Min Score: 0.60876775
Max Score: 0.80800545
